# <img align="left" src="./images/movie_camera.png"     style=" width:40px;  " > 练习实验：协同过滤推荐系统

在本练习中，你将实现协同过滤，构建一个电影推荐系统。 

# <img align="left" src="./images/film_reel.png"     style=" width:40px;  " > 大纲
- [ 1 - 符号约定](#1)
- [ 2 - 推荐系统](#2)
- [ 3 - 电影评分数据集](#3)
- [ 4 - 协同过滤学习算法](#4)
  - [ 4.1 协同过滤代价函数](#4.1)
    - [ 练习 1](#ex01)
- [ 5 - 学习电影推荐](#5)
- [ 6 - 推荐](#6)
- [ 7 - 恭喜！](#7)




##  包 <img align="left" src="./images/film_strip_vertical.png"     style=" width:40px;   " >
我们将使用现在已经熟悉的 NumPy 和 Tensorflow 包。

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from recsys_utils import *

<a name="1"></a>
## 1 - 符号约定


|通用 <br />  符号  | 描述| Python（如有） |
|:-------------|:------------------------------------------------------------||
| $r(i,j)$     | 标量；若用户 j 给电影 i 评过分则为 1，否则为 0             ||
| $y(i,j)$     | 标量；用户 j 给电影 i 的评分（在 r(i,j) = 1 时有定义） ||
|$\mathbf{w}^{(j)}$ | 向量；用户 j 的参数 ||
|$b^{(j)}$     |  标量；用户 j 的参数 ||
| $\mathbf{x}^{(i)}$ |   向量；电影 i 的特征值        ||     
| $n_u$        | 用户数量 |num_users|
| $n_m$        | 电影数量 | num_movies |
| $n$          | 特征数量 | num_features                    |
| $\mathbf{X}$ |  向量 $\mathbf{x}^{(i)}$ 组成的矩阵         | X |
| $\mathbf{W}$ |  向量 $\mathbf{w}^{(j)}$ 组成的矩阵         | W |
| $\mathbf{b}$ |  偏置参数 $b^{(j)}$ 组成的向量 | b |
| $\mathbf{R}$ | 元素为 $r(i,j)$ 的矩阵                    | R |



<a name="2"></a>
## 2 - 推荐系统 <img align="left" src="./images/film_rating.png" style=" width:40px;  " >
在本实验中，你将实现协同过滤学习算法，并将其应用于电影评分数据集。
协同过滤推荐系统的目标是生成两个向量：对每个用户，生成一个体现该用户电影品味的“参数向量”；对每部电影，生成一个同样大小的特征向量，体现对该电影的某种描述。两个向量的点积加上偏置项，就应该能估计出该用户可能给这部电影的评分。

下图详细展示了这些向量是如何学习出来的。

<figure>
   <img src="./images/ColabFilterLearn.PNG"  style="width:740px;height:250px;" >
</figure>

现有评分以矩阵形式给出，如图所示。$Y$ 包含评分：0.5 到 5（含端点），步长 0.5；未评分的电影为 0。$R$ 在已评分的位置为 1。电影在行上，用户在列上。每个用户有一个参数向量 $w^{user}$ 和偏置。每部电影有一个特征向量 $x^{movie}$。这些向量通过把现有的用户/电影评分作为训练数据同时学习出来。上面展示了一个训练样本：$\mathbf{w}^{(1)} \cdot \mathbf{x}^{(1)} + b^{(1)} = 4$。值得注意的是，特征向量 $x^{movie}$ 必须同时满足所有用户，而用户向量 $w^{user}$ 必须同时满足所有电影。这正是该方法名字的由来——所有用户协同生成这个评分集合。 

<figure>
   <img src="./images/ColabFilterUse.PNG"  style="width:640px;height:250px;" >
</figure>

特征向量和参数学习完成后，就可以用来预测用户对一部未评分电影可能给出的评分。如上图所示。图中的公式是为用户 1 预测电影 0 评分的例子。


在本练习中，你将实现函数 `cofiCostFunc` 来计算协同过滤的目标函数。实现目标函数之后，你将使用 TensorFlow 自定义训练循环来学习协同过滤的参数。第一步是详细介绍本实验将使用的数据集和数据结构。

<a name="3"></a>
## 3 - 电影评分数据集 <img align="left" src="./images/film_rating.png"     style=" width:40px;  " >
该数据集源自 [MovieLens "ml-latest-small"](https://grouplens.org/datasets/movielens/latest/) 数据集。   
[F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. <https://doi.org/10.1145/2827872>]

原始数据集包含 600 个用户对 9000 部电影的评分。数据集经过缩减，聚焦于 2000 年以来的电影。评分范围是 0.5 到 5，步长为 0.5。缩减后的数据集有 $n_u = 443$ 个用户和 $n_m= 4778$ 部电影。 

下面，你将把电影数据集加载到变量 $Y$ 和 $R$ 中。

矩阵 $Y$（一个 $n_m \times n_u$ 矩阵）存储评分 $y^{(i,j)}$。矩阵 $R$ 是一个二值指示矩阵，如果用户 $j$ 给电影 $i$ 评过分则 $R(i,j) = 1$，否则 $R(i,j)=0$。 

在本练习的这一部分中，你还将使用矩阵 $\mathbf{X}$、$\mathbf{W}$ 和 $\mathbf{b}$： 

$$\mathbf{X} = 
\begin{bmatrix}
--- (\mathbf{x}^{(0)})^T --- \\
--- (\mathbf{x}^{(1)})^T --- \\
\vdots \\
--- (\mathbf{x}^{(n_m-1)})^T --- \\
\end{bmatrix} , \quad
\mathbf{W} = 
\begin{bmatrix}
--- (\mathbf{w}^{(0)})^T --- \\
--- (\mathbf{w}^{(1)})^T --- \\
\vdots \\
--- (\mathbf{w}^{(n_u-1)})^T --- \\
\end{bmatrix},\quad
\mathbf{ b} = 
\begin{bmatrix}
 b^{(0)}  \\
 b^{(1)} \\
\vdots \\
b^{(n_u-1)} \\
\end{bmatrix}\quad
$$ 

$\mathbf{X}$ 的第 $i$ 行对应第 $i$ 部电影的特征向量 $x^{(i)}$，$\mathbf{W}$ 的第 $j$ 行对应第 $j$ 个用户的参数向量 $\mathbf{w}^{(j)}$。$x^{(i)}$ 和 $\mathbf{w}^{(j)}$ 都是 $n$ 维向量。在本练习中，你将使用 $n=10$，因此 $\mathbf{x}^{(i)}$ 和 $\mathbf{w}^{(j)}$ 有 10 个元素。相应地，$\mathbf{X}$ 是 $n_m \times 10$ 矩阵，$\mathbf{W}$ 是 $n_u \times 10$ 矩阵。

我们将从加载电影评分数据集开始，以理解数据的结构。
我们将用电影数据集加载 $Y$ 和 $R$。  
我们还会用预先计算好的值加载 $\mathbf{X}$、$\mathbf{W}$ 和 $\mathbf{b}$。这些值将在本实验后面学习出来，但这里我们先用预先计算的值来开发代价模型。

In [2]:
# 加载数据
X, W, b, num_movies, num_features, num_users = load_precalc_params_small()
Y, R = load_ratings_small()

print("Y", Y.shape, "R", R.shape)
print("X", X.shape)
print("W", W.shape)
print("b", b.shape)
print("num_features", num_features)
print("num_movies",   num_movies)
print("num_users",    num_users)

Y (4778, 443) R (4778, 443)
X (4778, 10)
W (443, 10)
b (1, 443)
num_features 10
num_movies 4778
num_users 443


In [3]:
#  从矩阵中，我们可以计算平均评分等统计量。
tsmean =  np.mean(Y[0, R[0, :].astype(bool)])
print(f"Average rating for movie 1 : {tsmean:0.3f} / 5" )

Average rating for movie 1 : 3.400 / 5


<a name="4"></a>
## 4 - 协同过滤学习算法 <img align="left" src="./images/film_filter.png"     style=" width:40px;  " >

现在，你将开始实现协同过滤学习算法。首先实现目标函数。 

在电影推荐场景下，协同过滤算法考虑一组 $n$ 维参数向量 $\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)}$、$\mathbf{w}^{(0)},...,\mathbf{w}^{(n_u-1)}$ 和 $b^{(0)},...,b^{(n_u-1)}$，其中模型对用户 $j$ 给电影 $i$ 的评分预测为 $y^{(i,j)} = \mathbf{w}^{(j)}\cdot \mathbf{x}^{(i)} + b^{(i)}$。给定一个由一些用户对一些电影给出的评分组成的数据集，你希望学习出能最佳拟合（最小化平方误差）的参数向量 $\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)}$、$\mathbf{w}^{(0)},...,\mathbf{w}^{(n_u-1)}$ 和 $b^{(0)},...,b^{(n_u-1)}$。

你将补全 cofiCostFunc 中的代码来计算协同过滤的代价函数。 


<a name="4.1"></a>
### 4.1 协同过滤代价函数

协同过滤代价函数为
$$J({\mathbf{x}^{(0)},...,\mathbf{x}^{(n_m-1)},\mathbf{w}^{(0)},b^{(0)},...,\mathbf{w}^{(n_u-1)},b^{(n_u-1)}})= \frac{1}{2}\sum_{(i,j):r(i,j)=1}(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2
+\underbrace{
\frac{\lambda}{2}
\sum_{j=0}^{n_u-1}\sum_{k=0}^{n-1}(\mathbf{w}^{(j)}_k)^2
+ \frac{\lambda}{2}\sum_{i=0}^{n_m-1}\sum_{k=0}^{n-1}(\mathbf{x}_k^{(i)})^2
}_{regularization}
\tag{1}$$
(1) 中第一个求和是“对所有满足 $r(i,j)$ 等于 $1$ 的 $i$、$j$ 求和”，也可以写成：

$$
= \frac{1}{2}\sum_{j=0}^{n_u-1} \sum_{i=0}^{n_m-1}r(i,j)*(\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)} - y^{(i,j)})^2
+\text{regularization}
$$

现在请你编写 cofiCostFunc（协同过滤代价函数）来返回这个代价。

<a name="ex01"></a>
### 练习 1

**for 循环实现：**   
先用 for 循环实现代价函数。
可以考虑分两步开发：先实现不含正则化的代价函数，下面提供了一个不含正则化的测试用例来检验你的实现。跑通之后，再加入正则化并运行包含正则化的测试。注意，只有当 $R(i,j) = 1$ 时，你才应该累加用户 $j$ 和电影 $i$ 对应的代价。

In [4]:
# GRADED FUNCTION: cofi_cost_func
# UNQ_C1

def cofi_cost_func(X, W, b, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    Args:
      X (ndarray (num_movies,num_features)): matrix of item features
      W (ndarray (num_users,num_features)) : matrix of user parameters
      b (ndarray (1, num_users)            : vector of user parameters
      Y (ndarray (num_movies,num_users)    : matrix of user ratings of movies
      R (ndarray (num_movies,num_users)    : matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
      lambda_ (float): regularization parameter
    Returns:
      J (float) : Cost
    """
    nm, nu = Y.shape
    J = 0
    # ## START CODE HERE ###  
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = X[i,:]
            y = Y[i,j]
            r = R[i,j]
            J += r * np.square((np.dot(w,x) + b_j - y ))
    J += (lambda_) * (np.sum(np.square(W)) + np.sum(np.square(X)))
    J = J/2
    # ## END CODE HERE ### 

    return J

In [5]:
# 公开测试
from public_tests import *
test_cofi_cost_func(cofi_cost_func)

All tests passed!


<details>
  <summary><font size="3" color="darkgreen"><b>点击查看提示</b></font></summary>
    你可以仿照 (1) 中的求和，用两个 for 循环来组织代码。   
    先实现不含正则化的代码。   
    注意 (1) 中有些元素是向量。使用 np.dot()。你也可以用 np.square()。
    密切注意哪些元素用 i 索引、哪些用 j 索引。别忘了除以 2。
    
```python     
    ### START CODE HERE ###  
    for j in range(nu):
        
        
        for i in range(nm):
            
            
    ### END CODE HERE ### 
```    
<details>
    <summary><font size="2" color="darkblue"><b> 点击查看更多提示</b></font></summary>
        
    这里有更多细节。下面的代码在使用矩阵元素之前先把它们逐一取出。 
    也可以直接引用矩阵。  
    这段代码不含正则化。
    
```python 
    nm,nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = 
            y = 
            r =
            J += 
    J = J/2
    ### END CODE HERE ### 

```
    
<details>
    <summary><font size="2" color="darkblue"><b>最后的手段（完整的非正则化实现）</b></font></summary>
    
```python 
    nm,nu = Y.shape
    J = 0
    ### START CODE HERE ###  
    for j in range(nu):
        w = W[j,:]
        b_j = b[0,j]
        for i in range(nm):
            x = X[i,:]
            y = Y[i,j]
            r = R[i,j]
            J += np.square(r * (np.dot(w,x) + b_j - y ) )
    J = J/2
    ### END CODE HERE ### 
```
    
<details>
    <summary><font size="2" color="darkblue"><b>正则化</b></font></summary>
     正则化就是把 W 数组和 X 数组的每个元素平方，再把所有平方项加起来。
     你可以使用 np.square() 和 np.sum()。

<details>
    <summary><font size="2" color="darkblue"><b>正则化细节</b></font></summary>
    
```python 
    J += lambda_* (np.sum(np.square(W)) + np.sum(np.square(X)))
```
    
</details>
</details>
</details>
</details>

    


In [6]:
# 缩小数据集规模以便更快运行
num_users_r = 4
num_movies_r = 5 
num_features_r = 3

X_r = X[:num_movies_r, :num_features_r]
W_r = W[:num_users_r,  :num_features_r]
b_r = b[0, :num_users_r].reshape(1,-1)
Y_r = Y[:num_movies_r, :num_users_r]
R_r = R[:num_movies_r, :num_users_r]

# 评估代价函数
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

Cost: 13.67


**预期输出（lambda = 0）**：  
$13.67$。

In [7]:
# 评估带正则化的代价函数 
J = cofi_cost_func(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

Cost (with regularization): 28.09


**预期输出**：

28.09

**向量化实现**

为计算 $J$ 创建向量化实现很重要，因为稍后优化过程中会多次调用它。本系列课程的重点不在于所用的线性代数，因此这里直接提供了实现。如果你是线性代数专家，也可以不看下面的代码，自己实现一版。 

运行下面的代码，验证它与非向量化版本产生相同的结果。

In [8]:
def cofi_cost_func_v(X, W, b, Y, R, lambda_):
    """
    Returns the cost for the content-based filtering
    Vectorized for speed. Uses tensorflow operations to be compatible with custom training loop.
    Args:
      X (ndarray (num_movies,num_features)): matrix of item features
      W (ndarray (num_users,num_features)) : matrix of user parameters
      b (ndarray (1, num_users)            : vector of user parameters
      Y (ndarray (num_movies,num_users)    : matrix of user ratings of movies
      R (ndarray (num_movies,num_users)    : matrix, where R(i, j) = 1 if the i-th movies was rated by the j-th user
      lambda_ (float): regularization parameter
    Returns:
      J (float) : Cost
    """
    j = (tf.linalg.matmul(X, tf.transpose(W)) + b - Y)*R
    J = 0.5 * tf.reduce_sum(j**2) + (lambda_/2) * (tf.reduce_sum(X**2) + tf.reduce_sum(W**2))
    return J

In [9]:
# 评估代价函数
J = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 0);
print(f"Cost: {J:0.2f}")

# 评估带正则化的代价函数 
J = cofi_cost_func_v(X_r, W_r, b_r, Y_r, R_r, 1.5);
print(f"Cost (with regularization): {J:0.2f}")

Cost: 13.67
Cost (with regularization): 28.09


**预期输出**：  
Cost: 13.67  
Cost (with regularization): 28.09

<a name="5"></a>
## 5 - 学习电影推荐 <img align="left" src="./images/film_man_action.png" style=" width:40px;  " >
------------------------------

完成协同过滤代价函数的实现之后，你就可以开始训练算法，让它为你做电影推荐了。 

在下面的单元格中，你可以输入自己的电影选择，算法随后会为你生成推荐！我们按自己的偏好填了一些值，但在用我们的选择跑通之后，你应该把它改成符合你口味的。
数据集中所有电影的列表在文件 [movie list](data/small_movie_list.csv) 中。

In [10]:
movieList, movieList_df = load_Movie_List_pd()

my_ratings = np.zeros(num_movies)          #  Initialize my ratings

# 查看 small_movie_list.csv 文件，找到我们数据集中每部电影的 id
# 例如，Toy Story 3 (2010) 的 ID 是 2700，所以要给它评“5 分”，可以设置
my_ratings[2700] = 5 

# 或者假设你不喜欢 Persuasion (2007)，可以设置
my_ratings[2609] = 2;

# 我们选了几部喜欢/不喜欢的电影，给出的评分
# 如下：
my_ratings[929]  = 5   # Lord of the Rings: The Return of the King, The
my_ratings[246]  = 5   # Shrek (2001)
my_ratings[2716] = 3   # Inception
my_ratings[1150] = 5   # Incredibles, The (2004)
my_ratings[382]  = 2   # Amelie (Fabuleux destin d'Amélie Poulain, Le)
my_ratings[366]  = 5   # Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
my_ratings[622]  = 5   # Harry Potter and the Chamber of Secrets (2002)
my_ratings[988]  = 3   # Eternal Sunshine of the Spotless Mind (2004)
my_ratings[2925] = 1   # Louis Theroux: Law & Disorder (2008)
my_ratings[2937] = 1   # Nothing to Declare (Rien à déclarer)
my_ratings[793]  = 5   # Pirates of the Caribbean: The Curse of the Black Pearl (2003)
my_rated = [i for i in range(len(my_ratings)) if my_ratings[i] > 0]

print('\nNew user ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0 :
        print(f'Rated {my_ratings[i]} for  {movieList_df.loc[i,"title"]}');


New user ratings:



Rated 5.0 for  Shrek (2001)
Rated 5.0 for  Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Rated 2.0 for  Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)
Rated 5.0 for  Harry Potter and the Chamber of Secrets (2002)
Rated 5.0 for  Pirates of the Caribbean: The Curse of the Black Pearl (2003)
Rated 5.0 for  Lord of the Rings: The Return of the King, The (2003)
Rated 3.0 for  Eternal Sunshine of the Spotless Mind (2004)
Rated 5.0 for  Incredibles, The (2004)
Rated 2.0 for  Persuasion (2007)
Rated 5.0 for  Toy Story 3 (2010)
Rated 3.0 for  Inception (2010)
Rated 1.0 for  Louis Theroux: Law & Disorder (2008)
Rated 1.0 for  Nothing to Declare (Rien à déclarer) (2010)


现在，让我们把这些评分加入 $Y$ 和 $R$，并对评分进行归一化。

In [11]:
# 重新加载评分并加入新评分
Y, R = load_ratings_small()
Y    = np.c_[my_ratings, Y]
R    = np.c_[(my_ratings != 0).astype(int), R]

# 归一化数据集
Ynorm, Ymean = normalizeRatings(Y, R)

让我们准备训练模型。初始化参数并选择 Adam 优化器。

In [12]:
#  常用的值
num_movies, num_users = Y.shape
num_features = 100

# 设置初始参数 (W, X)，用 tf.Variable 跟踪这些变量
tf.random.set_seed(1234) # for consistent results
W = tf.Variable(tf.random.normal((num_users,  num_features),dtype=tf.float64),  name='W')
X = tf.Variable(tf.random.normal((num_movies, num_features),dtype=tf.float64),  name='X')
b = tf.Variable(tf.random.normal((1,          num_users),   dtype=tf.float64),  name='b')

# 实例化一个优化器。
optimizer = keras.optimizers.Adam(learning_rate=1e-1)

现在让我们训练协同过滤模型。这将学习参数 $\mathbf{X}$、$\mathbf{W}$ 和 $\mathbf{b}$。 

同时学习 $w$、$b$ 和 $x$ 所涉及的运算，并不属于 TensorFlow 神经网络包提供的典型“层”。因此，课程 2 中的流程——Model、Compile()、Fit()、Predict()——不能直接套用。取而代之，我们可以使用自定义训练循环。

回忆一下之前实验中梯度下降的步骤。
- 重复直到收敛：
    - 计算前向传播
    - 计算损失相对于各参数的导数
    - 用学习率和算出的导数更新参数 
    
TensorFlow 拥有为你计算导数的强大能力，如下所示。在 `tf.GradientTape()` 代码段内，对 Tensorflow 变量的运算会被记录下来。之后调用 `tape.gradient()` 时，它会返回损失相对于被跟踪变量的梯度。然后就可以用优化器把梯度应用到参数上。 
这只是对 TensorFlow 及其他机器学习框架中一个实用功能的简要介绍。想了解更多，可以在你感兴趣的框架中查阅“自定义训练循环”（custom training loops）。
    


In [13]:
iterations = 200
lambda_ = 1
for iter in range(iterations):
    # 使用 TensorFlow 的 GradientTape
    # 记录用于计算代价的运算 
    with tf.GradientTape() as tape:

        # 计算代价（前向传播包含在代价计算中）
        cost_value = cofi_cost_func_v(X, W, b, Ynorm, R, lambda_)

    # 用梯度带自动求取
    # 可训练变量相对于损失的梯度
    grads = tape.gradient( cost_value, [X,W,b] )

    # 通过更新变量的值
    # 执行一步梯度下降，以最小化损失。
    optimizer.apply_gradients( zip(grads, [X,W,b]) )

    # 定期记录日志。
    if iter % 20 == 0:
        print(f"Training loss at iteration {iter}: {cost_value:0.1f}")

Training loss at iteration 0: 2321191.3


Training loss at iteration 20: 136169.3


Training loss at iteration 40: 51863.7


Training loss at iteration 60: 24599.0


Training loss at iteration 80: 13630.6


Training loss at iteration 100: 8487.7


Training loss at iteration 120: 5807.8


Training loss at iteration 140: 4311.6


Training loss at iteration 160: 3435.3


Training loss at iteration 180: 2902.1


<a name="6"></a>
## 6 - 推荐
下面，我们计算所有电影和用户的评分，并展示被推荐的电影。这些推荐基于上面在 `my_ratings[]` 中输入的电影和评分。要预测用户 $j$ 对电影 $i$ 的评分，计算 $\mathbf{w}^{(j)} \cdot \mathbf{x}^{(i)} + b^{(j)}$ 即可。用矩阵乘法可以一次性算出所有评分。

In [14]:
# 用训练好的权重和偏置进行预测
p = np.matmul(X.numpy(), np.transpose(W.numpy())) + b.numpy()

# 还原均值
pm = p + Ymean

my_predictions = pm[:,0]

# 对预测结果排序
ix = tf.argsort(my_predictions, direction='DESCENDING')

for i in range(17):
    j = ix[i]
    if j not in my_rated:
        print(f'Predicting rating {my_predictions[j]:0.2f} for movie {movieList[j]}')

print('\n\nOriginal vs Predicted ratings:\n')
for i in range(len(my_ratings)):
    if my_ratings[i] > 0:
        print(f'Original {my_ratings[i]}, Predicted {my_predictions[i]:0.2f} for {movieList[i]}')

Predicting rating 4.49 for movie My Sassy Girl (Yeopgijeogin geunyeo) (2001)
Predicting rating 4.48 for movie Martin Lawrence Live: Runteldat (2002)
Predicting rating 4.48 for movie Memento (2000)
Predicting rating 4.47 for movie Delirium (2014)
Predicting rating 4.47 for movie Laggies (2014)
Predicting rating 4.47 for movie One I Love, The (2014)
Predicting rating 4.46 for movie Particle Fever (2013)
Predicting rating 4.45 for movie Eichmann (2007)
Predicting rating 4.45 for movie Battle Royale 2: Requiem (Batoru rowaiaru II: Chinkonka) (2003)
Predicting rating 4.45 for movie Into the Abyss (2011)


Original vs Predicted ratings:

Original 5.0, Predicted 4.90 for Shrek (2001)
Original 5.0, Predicted 4.84 for Harry Potter and the Sorcerer's Stone (a.k.a. Harry Potter and the Philosopher's Stone) (2001)
Original 2.0, Predicted 2.13 for Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)
Original 5.0, Predicted 4.88 for Harry Potter and the Chamber of Secrets (2002)
Original 5.0, Predic

在实践中，可以利用附加信息来增强我们的预测。上面，前几百部电影的预测评分落在一个很小的区间内。我们可以在上述基础上进一步筛选：从这些高分电影中，挑出平均评分较高且评分数超过 20 的电影。本节使用 [Pandas](https://pandas.pydata.org/) 数据框，它有许多方便的排序功能。

In [15]:
filter=(movieList_df["number of ratings"] > 20)
movieList_df["pred"] = my_predictions
movieList_df = movieList_df.reindex(columns=["pred", "mean rating", "number of ratings", "title"])
movieList_df.loc[ix[:300]].loc[filter].sort_values("mean rating", ascending=False)

,pred,mean rating,number of ratings,title
1743,4.030961,4.252336,107,"Departed, The (2006)"
2112,3.985281,4.238255,149,"Dark Knight, The (2008)"
211,4.477798,4.122642,159,Memento (2000)
929,4.887054,4.118919,185,"Lord of the Rings: The Return of the King, The..."
2700,4.796531,4.109091,55,Toy Story 3 (2010)
653,4.357304,4.021277,188,"Lord of the Rings: The Two Towers, The (2002)"
1122,4.004471,4.006494,77,Shaun of the Dead (2004)
1841,3.980649,4.000000,61,Hot Fuzz (2007)
3083,4.084643,3.993421,76,"Dark Knight Rises, The (2012)"
2804,4.434171,3.989362,47,Harry Potter and the Deathly Hallows: Part 1 (...


<a name="7"></a>
## 7 - 恭喜！ <img align="left" src="./images/film_award.png"     style=" width:40px;  " >
你已经实现了一个实用的推荐系统！